In [2]:
import os
import pandas as pd
from urllib.parse import urlparse, unquote, parse_qs, urlunparse, urlencode
import re
import numpy as np
from tqdm import tqdm

tqdm.pandas()

chunks_dir = '../../datasets/alexa-top-1m/chunks/enriched'

In [ ]:
chunks = []

for chunk_file_path in os.listdir(chunks_dir):
  chunks.append(pd.read_csv(f'{chunks_dir}/{chunk_file_path}', index_col=0))

df = pd.concat(chunks, ignore_index=True)

df.info()

df.head()

<class 'pandas.DataFrame'>
RangeIndex: 714866 entries, 0 to 714865
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   url           714866 non-null  str  
 1   enriched_url  714866 non-null  str  
dtypes: str(2)
memory usage: 197.4 MB


,url,enriched_url
0,https://royal-lab.net/,"{'https://royal-lab.net/about-us/', 'https://r..."
1,https://creepypastafromthecrypt.blogspot.com/,{'https://creepypastafromthecrypt.blogspot.com...
2,https://www.gosnowmass.com/,{'https://www.gosnowmass.com/activity/uphill-s...
3,https://heilsarmee.ch/,{'https://heilsarmee.ch/themen-angebote/obdach...
4,https://cineten.com/,set()


In [ ]:
df['enriched_url'] = df['enriched_url'].progress_apply(eval)

100%|██████████| 714866/714866 [00:28<00:00, 25398.35it/s]


In [116]:
crawled_urls = set().union(*df['enriched_url'])

In [117]:
crawled_urls

{'https://www.ailr.it/bomboniere-solidali/',
 'http://hackeducation.com/2022/03/08/hope',
 'https://theplayground.co.uk/brian-eno-and-beatie-wolfe-transmit-new-album-liminal-into-space/',
 'https://www.cercle-enseignement.com/Primaire/Toute-l-actualite',
 'https://careers.lausd.org/pc/go/Communications-Careers/9596800/',
 'https://dpa-international.com/economics/urn:newsml:dpa.com:20090101:260207-99-414949',
 'https://hondasae.com/entry2/',
 'https://cityofholland.com/261/Public-Safety',
 'https://www.deece.edu.gr/forum/feed.php?mode=forums',
 'https://mantech.co.za/Stock.aspx?Query=PRE+INSULATEDand',
 'https://demland.pl/recenzje-filmowe-dema',
 'https://abc-gid.ru/author/markavtsov-flur25/',
 'https://www.sp.com.pg/our-brands',
 'https://shopmegadj.com/account/login',
 'https://www.fbbva.es/publicaciones/libro-enfermedades-alergicas/',
 'https://www.kitamura-print.com/print/panel/',
 'https://y-eg.jp/category/midokoro/',
 'https://expireddomains.com/domain/rpbo.com/backlink?hash=b258

In [ ]:
domains = set(urlparse(url).netloc for url in crawled_urls)

In [119]:
domains

{'e-bunpou.net',
 'alternative-doctor.com',
 'www.wopc.co.uk',
 'www.cinemagay.it',
 'www.fruugo.dk',
 'mpmb.in',
 'www.rintajouppi.fi',
 'www.rtl-theme.com',
 'www.liwenbianji.cn',
 '1post.jp',
 'www.superecono.com',
 'theclearbrands.com',
 'pokercopilot.com',
 'bottlelogic.com',
 'eng.sectsco.org',
 'modlitba.sk',
 'www.iquesta.com',
 'australianfintech.com.au',
 'www.princegeorge.ca',
 'www.foodiesfeed.com',
 'radioguestlist.com',
 'www.louie.com.br',
 'denovosoftware.com',
 'www.happyfoto.cz',
 'www.radio4u.in',
 'web-ace.jp',
 'afaa.com',
 'dnsomatic.com',
 'www.celoxis.com',
 'petelagi.com',
 'uasemena.com.ua',
 'www.totcar.com',
 'tribunaux-rechtbanken.be',
 'fatemi.ir',
 'www.smartspeechtherapy.com',
 'rafaelcarvalho.tv',
 'mof.gov.bt',
 'www.pakistanjobsbank.com',
 'www.cfpc.ca',
 'www.eligibilis.com',
 'www.gympos.sk',
 'mypage.dpub.jp',
 'www.airgunspares.com',
 'newsturk.ru',
 'cinesunstar.com',
 'mamoszurnalas.lt',
 'csociales.wordpress.com',
 'topjobs.lk',
 'www.rbl.bank.

In [121]:
crawled_df = pd.DataFrame({'url': list(crawled_urls)})

In [122]:
df = pd.concat([df[['url']], crawled_df], ignore_index=True)

In [123]:
df

,url
0,https://royal-lab.net/
1,https://creepypastafromthecrypt.blogspot.com/
2,https://www.gosnowmass.com/
3,https://heilsarmee.ch/
4,https://cineten.com/
...,...
3485251,https://csonline2.net/servers
3485252,https://www.aallnet.org/education-training/in-...
3485253,https://www.disney.es/peliculas/toy-story-5
3485254,https://honari.com/candleCal


In [124]:
df['url'].nunique()

3231343

In [ ]:
def strip_to_base_url(url):
    """
    Strip URL to just scheme + netloc (e.g., https://example.com/)
    """
    parsed = urlparse(url)
    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        '/',  # Just root path
        '',   # No params
        '',   # No query
        ''    # No fragment
    ))
    
print(
  strip_to_base_url('https://cast-ad.co.jp/works/%e5%8c%bb%e7%99%82%e6%b8%ac%e5%ae%9a%e5%99%a8%e3%81%a8%e9%80%a3%e5%8b%95%e3%81%97%e3%81%9f%e3%82%ab%e3%83%ab%e3%83%86%e3%82%a2%e3%83%97%e3%83%aa%e3%81%ae%e9%96%8b%e7%99%ba%ef%bc%9a%e5%8c%bb%e7%99%82/')
)

https://cast-ad.co.jp/


In [127]:
def strip_params(url):
    """
    Strip URL to just scheme + netloc (e.g., https://example.com/)
    """
    parsed = urlparse(url)
    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        '',   # No params
        '',   # No query
        ''    # No fragment
    ))

In [128]:
def remove_specific_params(url, params_to_remove):
    """
    Remove specific query parameters, keep all others
    Does NOT cascade - only removes the exact params specified
    """
    parsed = urlparse(url)
    
    if not parsed.params and not parsed.query:
        return url
    
    params = parse_qs(parsed.params, keep_blank_values=True)
    query = parse_qs(parsed.query, keep_blank_values=True)
    
    # Remove only specified params (case-insensitive)
    clean_params = {
        k: v for k, v in params.items()
        if all(p.lower() not in k.lower()  for p in params_to_remove)
    }
    
    clean_query = {
        k: v for k, v in query.items()
        if all(p.lower() not in k.lower()  for p in params_to_remove)
    }
    
    # Rebuild query
    new_params = urlencode(clean_params, doseq=True) if clean_params else ''

    new_query  = urlencode(clean_query, doseq=True)  if clean_query  else ''
    
    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        new_params,
        new_query,
        parsed.fragment
    ))
    
print(
  remove_specific_params(
    'https://chemnetbase.com/search/SimpleSearch.xhtml;jsessionid=FE7D77AC8225B27425FC97DF6097E119?dswid=-5046',
    {'utm_campaign', 'utm_medium', 'utm_source', 'sessionid'}
  )
)

https://chemnetbase.com/search/SimpleSearch.xhtml?dswid=-5046


In [ ]:
# 1. Domain parking/marketplace
parking_services = [
  'hugedomains.com', 'afternic.com', 'sedo.com', 'dropcatch.com',
  'domainmarket.com', 'brandbucket.com', 'expireddomains.com',
  'forsaledomain.net', 'dynadot.com', 'domain_profile.cfm'
]

# Tracking params to remove
TRACKING_PARAMS = {
  # Tracking
  'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_term',
  # Analytics
  'fbclid', 'gclid', '_ga', '_gac', 'mc_cid', 'mc_eid',
  # Session IDs
  'phpsessid', 'jsessionid', 'sessionid', 'sid', 'aspsessionid',
  # OAuth/Auth tokens (for Google accounts specifically)
  'state', 'nonce', 'code', 'client-request-id', 'wsucxt', 'claims',
  'response_type', 'response_mode', 'flowname', 'flowentry', 'passive',
  'ifkv', 'dsh', 'authuser', 'authError', 'client_id', 'continue',
  'followup', 'dsh', 'sv',
  # Misc tokens
  'uuid', 'vid', 'nk', 'client_key', 'cobrandid',
  # Referrals
  'sharename', 'ref', 'source', 'from', 'referrer',
}

def clean_url(url):
  url_lower = url.lower()
  
  parsed = urlparse(url)
  netloc = parsed.netloc.lower()
  url_clean = remove_specific_params(url, TRACKING_PARAMS)

  # sampling 1% of parking urls
  if any(ps in url_lower for ps in parking_services):
    if np.random.random() < 0.01:
      return url_clean
    else:
      return None

  # sampling 1% of tumblr urls
  if 'tumblr.com' in netloc and netloc not in ['www.tumblr.com', 'tumblr.com']:
    if np.random.random() < 0.01:
      return url_clean
    else:
      return None

  # stripping suspend/error/captcha pages to base
  if 'login.microsoftonline.com' in netloc or any(x in parsed.path.lower() for x in ['/suspendedpage', '/404.html', '/page-not-found', '/error']) or  any(x in url_lower for x in ['captcha', 'iamhuman', '/waf/verify', '/validation/', 'bot-check', 'bot_check']):
    if np.random.random() < 0.01:
      return url_clean
    else:
      return strip_to_base_url(url_clean)
  
  # 'accounts.google.com' urls
  if 'accounts.google.com' in netloc:
    return strip_params(url_clean)

  # removing ip addresses, we'll suppose they are all malicious because ip addresses look all the same
  if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', netloc):
    return None
  
  unquoted_url = unquote(url_clean)

  while unquoted_url != url_clean:
    url_clean = unquoted_url
    unquoted_url = unquote(url_clean)
    
  return unquoted_url

In [142]:
df['clean_url'] = df['url'].progress_apply(clean_url)

100%|██████████| 3485256/3485256 [01:29<00:00, 39136.13it/s]


In [ ]:
df.drop(df[df['url'].str.len() > 5000].index, inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
# df.loc[df['clean_url'].notna(), ['clean_url']].drop_duplicates(ignore_index=True).rename(columns={'clean_url': 'url'}).to_csv('../../datasets/benign.csv')

In [ ]:
# samples_df = pd.read_csv('../../datasets/benign.csv', index_col=0).sample(frac=0.3, ignore_index=True)

In [ ]:
# samples_df.to_csv('../../datasets/samples.csv')

In [8]:
df = pd.read_csv('../../datasets/benign.csv', index_col=0)

df

,url
0,https://royal-lab.net
1,https://creepypastafromthecrypt.blogspot.com
2,https://www.gosnowmass.com
3,https://heilsarmee.ch
4,https://cineten.com
...,...
3179075,https://csonline2.net/servers
3179076,https://www.aallnet.org/education-training/in-...
3179077,https://www.disney.es/peliculas/toy-story-5
3179078,https://honari.com/candleCal
